# SAP Procurement Fraud Detection Dashboard

This notebook is the visualization layer for the SAP Procurement Fraud Detection Analytics project. It consolidates duplicate invoice, split purchase, abnormal vendor, and approval bypass analytics into one executive dashboard.

**Business objective:** help Finance, Internal Audit, and SAP MM/FI control owners prioritize suspicious procurement exceptions by exposure, vendor, and control weakness.

## 1. Load dashboard builder

The reusable logic is stored in `fraud_dashboard.py` so the dashboard can be run from both this notebook and a command-line/GitHub Actions workflow. The builder uses Python standard-library data processing and writes an HTML dashboard that uses Plotly in the browser.

In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "03_Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DASHBOARD_SCRIPT = PROJECT_ROOT / "06. Visualization" / "fraud_dashboard.py"
spec = importlib.util.spec_from_file_location("fraud_dashboard", DASHBOARD_SCRIPT)
fraud_dashboard = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fraud_dashboard)

print(f"Project root: {PROJECT_ROOT}")

## 2. Build dashboard artifacts

Running `build_dashboard()` loads the CSV tables in `03_Data`, applies the fraud detection rules, and writes audit-friendly output files under `06. Visualization/outputs`.

In [ ]:
summary, exceptions, outputs = fraud_dashboard.build_dashboard(PROJECT_ROOT)
summary

## 3. Executive KPI dashboard

Open the generated HTML file for the fully interactive dashboard. It contains KPI cards, fraud-category exposure, exception mix, monthly trends, top vendors, and SAP control recommendations.

In [ ]:
dashboard_html = outputs["html"]
print(f"Dashboard generated: {dashboard_html.relative_to(PROJECT_ROOT)}")
display(HTML(f'<p><a href="{dashboard_html.relative_to(PROJECT_ROOT)}" target="_blank">Open interactive dashboard HTML</a></p>'))

## 4. Audit exception counts

These counts show how many detailed records support each dashboard KPI. The generated CSV files can be reviewed as audit workpapers.

In [ ]:
exception_counts = [
    {"exception_table": name, "record_count": len(rows)}
    for name, rows in exceptions.items()
]
exception_counts

## 5. Top exceptions for investigation

The samples below give management an immediate starting point for vendor or transaction follow-up.

In [ ]:
top_exception_samples = {}
for name, rows in exceptions.items():
    top_exception_samples[name] = rows[:5]

top_exception_samples

## 6. SAP control recommendations

Analytics should lead to process improvement. The generated control file links each fraud category to an SAP control action.

In [ ]:
outputs["controls"].read_text(encoding="utf-8").splitlines()[:6]

## 7. Output files

These files are generated automatically and can be committed or downloaded for review.

In [ ]:
for output_name, output_path in outputs.items():
    print(f"{output_name}: {output_path.relative_to(PROJECT_ROOT)}")